# Evaluation Pipeline and Baseline 2

 What accuracy and cost does each model achieve on clean factual QA? And how do different scoring strategies affect what we measure?



In [2]:
import os, asyncio, json, time, hashlib
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from anthropic import AsyncAnthropic
from dotenv import load_dotenv; load_dotenv("../../.env")

ac    = AsyncAnthropic()
CACHE = Path(".cache/baseline"); CACHE.mkdir(parents=True, exist_ok=True)

def cache_key(*parts) -> Path:
    h = hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:16]
    return CACHE / f"{h}.json"

async def call(prompt, model="claude-haiku-4-5-20251001", max_tokens=64):
    k = cache_key(prompt, model, max_tokens)
    if k.exists():
        r = json.loads(k.read_text()); r["cache_hit"] = True; return r
    t0 = time.time()
    r  = await ac.messages.create(model=model, max_tokens=max_tokens,
                                   messages=[{"role": "user", "content": prompt}])
    result = {"text": r.content[0].text, "in_tok": r.usage.input_tokens,
             "out_tok": r.usage.output_tokens, "latency": time.time()-t0, "cache_hit": False}
    k.write_text(json.dumps(result)); return result

print("Pipeline ready.")

Pipeline ready.


In [3]:
EVAL_SET = [
    # easy: single-hop
    {"qid": "e1", "difficulty": "easy",   "q": "What is the capital of France?",              "gold": "Paris"},
    {"qid": "e2", "difficulty": "easy",   "q": "What element has symbol Au?",                 "gold": "Gold"},
    {"qid": "e3", "difficulty": "easy",   "q": "How many planets are in the solar system?",   "gold": "8"},
    {"qid": "e4", "difficulty": "easy",   "q": "What year did WWII end?",                    "gold": "1945"},
    {"qid": "e5", "difficulty": "easy",   "q": "Who wrote Hamlet?",                          "gold": "Shakespeare"},
    {"qid": "e6", "difficulty": "easy",   "q": "What is the largest ocean?",                  "gold": "Pacific"},
    {"qid": "e7", "difficulty": "easy",   "q": "What is the boiling point of water in Celsius?","gold": "100"},
    {"qid": "e8", "difficulty": "easy",   "q": "What planet is closest to the Sun?",          "gold": "Mercury"},
    # hard: multi-hop or obscure
    {"qid": "h1", "difficulty": "hard",   "q": "What ocean does the Amazon drain into?",      "gold": "Atlantic"},
    {"qid": "h2", "difficulty": "hard",   "q": "What language is spoken where Machu Picchu is?", "gold": "Spanish"},
    {"qid": "h3", "difficulty": "hard",   "q": "What is the capital of the country with the Colosseum?", "gold": "Rome"},
    {"qid": "h4", "difficulty": "hard",   "q": "What gas do plants absorb during photosynthesis?","gold": "carbon dioxide"},
    {"qid": "h5", "difficulty": "hard",   "q": "Who painted the Sistine Chapel ceiling?",      "gold": "Michelangelo"},
    {"qid": "h6", "difficulty": "hard",   "q": "What is the chemical symbol for gold?",       "gold": "Au"},
    {"qid": "h7", "difficulty": "hard",   "q": "What is the atomic number of carbon?",         "gold": "6"},
    {"qid": "h8", "difficulty": "hard",   "q": "In what year did the Berlin Wall fall?",       "gold": "1989"},
]

In [4]:
# correctness scorer.
# This is not trivial — need to think about what strategy is fairest:
#   - exact match: strict but misses "Paris, France" when gold is "Paris"
#   - substring: lenient but might over-count ("6" matches "26")
#   - normalise first: strip punctuation, lowercase, collapse whitespace
#
# Implementing THREE strategies and we'll compare how they differ on the eval set.

import re

def normalise(s: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace."""
    s = s.lower()
    s = re.sub(r'[^\w\s]', '', s)
    s = s.strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def exact_match(gold: str, response: str) -> bool:
    return normalise(gold) == normalise(response)

def substring_match(gold: str, response: str) -> bool:
    return normalise(gold) in normalise(response)

def token_f1(gold: str, response: str) -> float:
    """SQuAD-style F1: overlap of token sets between gold and response."""
    gold_tokens = set(normalise(gold).split())
    response_tokens = set(normalise(response).split())
    if not gold_tokens and not response_tokens:
        return 1.0
    if not gold_tokens or not response_tokens:
        return 0.0
    intersection = gold_tokens & response_tokens
    precision = len(intersection) / len(response_tokens)
    recall = len(intersection) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

# Tests
assert exact_match("Paris",  "Paris"),               "exact: same string"
assert not exact_match("Paris", "Paris, France"),    "exact: no partial"
assert substring_match("Paris", "The answer is Paris, France."), "substr: partial ok"
assert 0 < token_f1("New York City", "New York") < 1, "f1: partial overlap"
print("All scorer tests passed")

All scorer tests passed


In [5]:
# Run the eval and compare all three scoring strategies
async def run_eval():
    async def _one(item):
        prompt = f"Answer in 1-3 words only: {item['q']}"
        r = await call(prompt)
        print(r)
        return {
            "qid": item["qid"], "difficulty": item["difficulty"],
            "gold": item["gold"], "response": r["text"],
            "em":   exact_match(item["gold"],    r["text"]),
            "sub":  substring_match(item["gold"], r["text"]),
            "f1":   token_f1(item["gold"],        r["text"]),
            "input_tokens": r["in_tok"], "output_tokens": r["out_tok"],
            "latency": r["latency"],
        }
    return await asyncio.gather(*[_one(item) for item in EVAL_SET])

rows = await run_eval()
df   = pd.DataFrame(rows)
print(df[["qid","difficulty","gold","response","em","sub","f1"]].to_string(index=False))

{'text': 'Paris', 'in_tok': 24, 'out_tok': 4, 'latency': 1.370577096939087, 'cache_hit': True}
{'text': 'Gold', 'in_tok': 23, 'out_tok': 4, 'latency': 1.1545689105987549, 'cache_hit': True}
{'text': 'Eight planets.', 'in_tok': 26, 'out_tok': 6, 'latency': 2.1903738975524902, 'cache_hit': True}
{'text': '1945', 'in_tok': 25, 'out_tok': 6, 'latency': 1.5086688995361328, 'cache_hit': True}
{'text': 'William Shakespeare', 'in_tok': 22, 'out_tok': 5, 'latency': 1.2280120849609375, 'cache_hit': True}
{'text': 'Pacific Ocean.', 'in_tok': 23, 'out_tok': 6, 'latency': 1.9819960594177246, 'cache_hit': True}
{'text': '100°C', 'in_tok': 29, 'out_tok': 7, 'latency': 3.0242090225219727, 'cache_hit': True}
{'text': 'Mercury', 'in_tok': 25, 'out_tok': 4, 'latency': 1.4186770915985107, 'cache_hit': True}
{'text': 'Atlantic Ocean.', 'in_tok': 25, 'out_tok': 6, 'latency': 2.292901039123535, 'cache_hit': True}
{'text': 'Spanish, Quechua', 'in_tok': 30, 'out_tok': 9, 'latency': 1.944244146347046, 'cache_hi

In [ ]:
# Compute cost-per-correct for each scoring strategy.
# Cost = (input_tokens × in_price + output_tokens × out_price) / 1_000_000
# Haiku pricing: input $0.80/M, output $4.00/M.. need to update 
#
# Then: which questions does substring_match count as correct that exact_match misses?

IN_PRICE, OUT_PRICE = 0.80, 4.00   # USD per million tokens

def cost_per_correct(df, score_col):
    """Return (total_cost_usd, cost_per_correct_usd) for the given score column."""
    total_cost = 0
    correct_count = 0
    for _, row in df.iterrows():
        if row[score_col]:
            total_cost += (row["input_tokens"] * IN_PRICE + row["output_tokens"] * OUT_PRICE) / 1_000_000
            correct_count += 1
    return total_cost, total_cost / correct_count if correct_count > 0 else 0

for metric in ["em", "sub", "f1"]:
    total, cpc = cost_per_correct(df, metric)
    acc = df[metric].mean() if metric != "f1" else df[metric].mean()
    print(f"{metric:3s}  acc={acc:.2f}  total_cost=${total*1000:.3f}m  cost_per_correct=${cpc*1000:.3f}m")

print("\nQuestions where substring passes but exact_match fails:")
print(df[df["sub"] & ~df["em"]][["qid", "difficulty", "gold", "response"]])


# lets see how much accracy differs between the easy and hard questions under each metric:
for metric in ["em", "sub", "f1"]:
    easy_acc = df[df["difficulty"] == "easy"][metric].mean()
    hard_acc = df[df["difficulty"] == "hard"][metric].mean()
    print(f"{metric:3s}  easy_acc={easy_acc:.2f}  hard_acc={hard_acc:.2f}")



# lets also find how many questions wrong are withing one standard deviation to account for pure variance rather than systematic error:
# note thatt this is found by equation: std = sqrt(p*(1-p)/n) where p is the accuracy and n is the number of samples (8 in this case)
for metric in ["em", "sub", "f1"]:
    accs = []
    for _ in range(1000):
        sample = df.sample(frac=1, replace=True)  # bootstrap resample
        acc = sample[metric].mean() if metric != "f1" else sample[metric].mean()
        accs.append(acc)
    mean_acc = sum(accs) / len(accs)
    std_acc = (sum((x - mean_acc) ** 2 for x in accs) / len(accs)) ** 0.5
    print(f"{metric:3s}  acc={mean_acc:.2f} ± {std_acc:.2f}")

em   acc=0.56  total_cost=$0.370m  cost_per_correct=$0.041m
sub  acc=0.94  total_cost=$0.658m  cost_per_correct=$0.044m
f1   acc=0.77  total_cost=$0.607m  cost_per_correct=$0.043m

Questions where substring passes but exact_match fails:
   qid difficulty         gold             response
4   e5       easy  Shakespeare  William Shakespeare
5   e6       easy      Pacific       Pacific Ocean.
6   e7       easy          100                100°C
8   h1       hard     Atlantic      Atlantic Ocean.
9   h2       hard      Spanish     Spanish, Quechua
10  h3       hard         Rome         Rome, Italy.
em   easy_acc=0.50  hard_acc=0.62
sub  easy_acc=0.88  hard_acc=1.00
f1   easy_acc=0.67  hard_acc=0.88
em   acc=0.56 ± 0.12
sub  acc=0.94 ± 0.06
f1   acc=0.77 ± 0.08


## Findings

the metric MASSIVELY affects the accuracy - it is one of the hardest aspects of testing. a lot of edge cases to consider, again especially to multi worded answers and numerical answers. 

for the main phase diagram F1 + substring as secondary sanity check is ideal. although again, probably the less restrictive metrics are in this case (case where we test for facts) better, as usually in these scenarios even one similar word/number indicates correctness. 

variations between easy and hard are less extreme, but still a 10+ percentage point which is semi within one std -- interesting to note that HARD questions score higher on accuracy in general 

note also that in some cases distractors fundamentally change the structuring of the answers ("It might be Paris..."), so need to account for that. eg in this case, a substring might be a better metric for those reasons. 

also for costs - should be measuring how much we spend before we get the right answer. should be careful there